# Fine-tuning NLLB-200 pour le Mooré (anglais <-> mooré)

Ce notebook fait partie du projet d'app de traduction IA (Mooré, Dioula, Bambara + langues internationales).
Objectif : améliorer la traduction anglais<->mooré du modèle NLLB-200, qui échoue actuellement sur certaines phrases
(dégénérescence par répétition observée sur le baseline — voir le journal d'apprentissage du projet).

**Pourquoi anglais et pas français ?** La seule ressource texte Mooré propre légalement (Google SMOL, licence CC-BY-4.0,
réutilisable commercialement) n'existe qu'en paires anglais-mooré, pas français-mooré. Pour l'app, on route donc le
français vers le mooré EN PASSANT PAR l'anglais (français -> anglais, déjà excellent nativement avec NLLB ; puis
anglais <-> mooré, amélioré ici). C'est une stratégie de "langue pivot", classique quand on manque de données directes
entre deux langues mais qu'on a de bonnes données via une langue intermédiaire.

**Avant de lancer ce notebook** : Menu `Exécution` (Runtime) -> `Modifier le type d'exécution` -> Accélérateur matériel
-> **T4 GPU** (gratuit). Sans ça, l'entraînement tournera sur CPU et sera beaucoup trop lent.

## 1. Installation des librairies

On indique une version MINIMALE pour chaque librairie plutôt qu'une version exacte figée : ça laisse pip choisir
une version dont le "wheel" (paquet pré-compilé) existe pour la version de Python de Colab. Une version exacte
trop ancienne peut n'avoir aucun wheel pré-compilé disponible pour un Python plus récent, ce qui force pip à
recompiler la librairie depuis son code source — une opération fragile qui échoue souvent (c'est ce qui se passait
avec la version figée précédente : échec de compilation de `tokenizers`, une dépendance de `transformers` écrite
en Rust). Compromis assumé : un peu moins de reproductibilité parfaite, en échange d'un notebook qui s'installe
correctement aujourd'hui, sur l'environnement Colab du moment.

On met aussi à jour `torchao` (une librairie de quantization qu'on n'utilise pas directement, mais dont `peft`
vérifie la version dès qu'elle est présente sur le système). Colab la préinstalle en usine dans une version trop
ancienne pour les versions récentes de `peft` — sans cette mise à jour, `get_peft_model()` refuse de démarrer.

In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade torchao
!pip install -q "transformers>=4.44" "peft>=0.13" "accelerate>=0.34" "datasets>=2.21" "sacrebleu>=2.4" "huggingface_hub>=0.24"

## 2. Préparation des données

On reconstruit ici le même corpus que celui préparé en local pour le projet (voir
`backend/scripts/prepare_moore_data.py`) : trois sous-corpus Google SMOL (tous CC-BY-4.0, donc réutilisables
commercialement, contrairement à la plupart des autres ressources trouvées pour le Mooré) :
- `smolsent` : phrases traduites professionnellement (unité = 1 phrase)
- `smoldoc` : documents entiers traduits, qu'on aplatit en phrases individuelles
- `gatitos` : lexique mot-à-mot / courtes expressions

**Train/validation split** : le jeu de validation (200 paires, jamais montrées pendant l'entraînement) sert à mesurer
si le modèle généralise à des phrases nouvelles, plutôt que de simplement mémoriser ses exemples d'entraînement
(surapprentissage / overfitting).

In [ ]:
import json, random

from huggingface_hub import hf_hub_download



random.seed(42)



def load_smolsent():

    path = hf_hub_download(repo_id='google/smol', filename='smolsent/en_mos.jsonl', repo_type='dataset')

    with open(path, encoding='utf-8') as f:

        return [{'en': json.loads(l)['src'], 'mos': json.loads(l)['trg']} for l in f]



def load_smoldoc():

    path = hf_hub_download(repo_id='google/smol', filename='smoldoc/en_mos.jsonl', repo_type='dataset')

    pairs = []

    with open(path, encoding='utf-8') as f:

        for line in f:

            entry = json.loads(line)

            for s, t in zip(entry['srcs'], entry['trgs']):

                pairs.append({'en': s, 'mos': t})

    return pairs



def load_gatitos():

    path = hf_hub_download(repo_id='google/smol', filename='gatitos/en_mos.jsonl', repo_type='dataset')

    pairs = []

    with open(path, encoding='utf-8') as f:

        for line in f:

            entry = json.loads(line)

            if entry['trgs']:

                pairs.append({'en': entry['src'], 'mos': entry['trgs'][0]})

    return pairs



all_pairs = load_smolsent() + load_smoldoc() + load_gatitos()



seen, unique_pairs = set(), []

for pair in all_pairs:

    en, mos = pair['en'].strip(), pair['mos'].strip()

    if not en or not mos:

        continue

    key = (en, mos)

    if key not in seen:

        seen.add(key)

        unique_pairs.append({'en': en, 'mos': mos})



random.shuffle(unique_pairs)

val_pairs = unique_pairs[:200]

train_pairs = unique_pairs[200:]

print(f'Entrainement: {len(train_pairs)} paires, Validation: {len(val_pairs)} paires')

print('Exemple:', train_pairs[0])

## 3. Chargement du modèle de base + LoRA

**LoRA (Low-Rank Adaptation)** : au lieu de ré-entraîner les 600 millions de paramètres du modèle NLLB (lent, gourmand
en mémoire, et risqué avec seulement ~6000 phrases d'entraînement — le modèle pourrait "oublier" ses autres langues,
un phénomène appelé *oubli catastrophique*), on gèle le modèle de base et on n'entraîne que de petites matrices
additionnelles insérées dans les couches d'attention (quelques millions de paramètres seulement, souvent <1% du
total). Résultat : entraînement beaucoup plus rapide, tient largement dans la mémoire d'un GPU gratuit (T4, 16 Go),
et le modèle de base reste intact pour toutes les autres langues du projet.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from peft import LoraConfig, get_peft_model, TaskType



MODEL_NAME = 'facebook/nllb-200-distilled-600M'



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)



lora_config = LoraConfig(

    task_type=TaskType.SEQ_2_SEQ_LM,

    r=16,                 # dimension des matrices LoRA : plus grand = plus de capacité, plus lent

    lora_alpha=32,        # facteur d'echelle de l'adaptation (convention courante: 2x r)

    lora_dropout=0.05,

    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],  # couches d'attention à adapter

)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()  # affiche le % de paramètres réellement entraînés

## 4. Tokenisation

Astuce spécifique à NLLB : la langue "active" du tokenizer (`tokenizer.src_lang`) détermine quel jeton de langue
est ajouté au DÉBUT du texte encodé. On l'utilise donc deux fois avec des valeurs différentes : une fois pour encoder
le texte source (anglais), une fois pour encoder le texte cible qui devient les `labels` (le mooré) — c'est ce qui
garantit que la séquence cible commence bien par le jeton `mos_Latn`, exactement comme `forced_bos_token_id` le fait
à l'inférence (voir `backend/app/translation_service.py`).

In [ ]:
from datasets import Dataset



train_dataset = Dataset.from_list(train_pairs)

val_dataset = Dataset.from_list(val_pairs)



MAX_LENGTH = 128



def preprocess(examples):

    tokenizer.src_lang = 'eng_Latn'

    model_inputs = tokenizer(examples['en'], max_length=MAX_LENGTH, truncation=True)



    tokenizer.src_lang = 'mos_Latn'

    labels = tokenizer(examples['mos'], max_length=MAX_LENGTH, truncation=True)



    model_inputs['labels'] = labels['input_ids']

    return model_inputs



train_tokenized = train_dataset.map(preprocess, batched=True, remove_columns=['en', 'mos'])

val_tokenized = val_dataset.map(preprocess, batched=True, remove_columns=['en', 'mos'])

## 5. Mesure AVANT fine-tuning (baseline)

On traduit les 200 phrases de validation avec le modèle de base (pas encore fine-tuné) et on compare au mooré de
référence avec la métrique **chrF** (F-score sur les n-grammes de caractères, pas de mots — plus adaptée que BLEU
pour les langues morphologiquement complexes/peu dotées comme le Mooré, où BLEU pénalise trop sévèrement les petites
variations orthographiques). Ce chiffre sert de point de comparaison : le fine-tuning a-t-il vraiment aidé ?

In [ ]:
import torch

import sacrebleu



def translate_batch(model_to_use, texts, batch_size=16):

    model_to_use.eval()

    tokenizer.src_lang = 'eng_Latn'

    outputs = []

    forced_bos = tokenizer.convert_tokens_to_ids('mos_Latn')

    device = next(model_to_use.parameters()).device

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH).to(device)

        with torch.no_grad():

            generated = model_to_use.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LENGTH, num_beams=5)

        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    return outputs



if torch.cuda.is_available():

    model = model.to('cuda')



val_sources = [p['en'] for p in val_pairs]

val_references = [p['mos'] for p in val_pairs]



baseline_predictions = translate_batch(model, val_sources)

baseline_chrf = sacrebleu.corpus_chrf(baseline_predictions, [val_references])

print(f'chrF AVANT fine-tuning : {baseline_chrf.score:.2f}')

print('Exemple :', val_sources[0], '->', baseline_predictions[0], '(reference:', val_references[0], ')')

## 6. Entraînement

Paramètres choisis pour ce cas précis (petit jeu de données, LoRA) :
- `learning_rate=1e-4` : plus élevé que pour un fine-tuning complet (souvent 1e-5/2e-5), car on n'entraîne qu'une
  petite fraction des poids — ils peuvent bouger plus vite sans déstabiliser le reste du modèle.
- `num_train_epochs=10` : une *epoch* = un passage complet sur tout le jeu d'entraînement. Avec seulement ~6000
  phrases, plusieurs passages sont nécessaires pour que le modèle apprenne un signal utile.
- `load_best_model_at_end=True` avec évaluation à chaque epoch : garde automatiquement la version du modèle qui a
  le mieux généralisé sur la validation, pas forcément la dernière (protection simple contre le surapprentissage).

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

import numpy as np



data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)



def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels])

    return {'chrf': chrf.score}



training_args = Seq2SeqTrainingArguments(

    output_dir='./moore-nllb-lora-checkpoints',

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=1e-4,

    num_train_epochs=10,

    eval_strategy='epoch',

    save_strategy='epoch',

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model='chrf',

    predict_with_generate=True,

    generation_max_length=MAX_LENGTH,

    fp16=torch.cuda.is_available(),  # calcul en demi-precision sur GPU : ~2x plus rapide, negligeable perte de qualite

    report_to='none',

)



trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

)



trainer.train()

## 7. Mesure APRÈS fine-tuning

Même mesure chrF qu'à l'étape 5, avec le modèle maintenant fine-tuné, sur exactement les mêmes phrases de
validation — comparaison directe et équitable.

In [ ]:
finetuned_predictions = translate_batch(model, val_sources)

finetuned_chrf = sacrebleu.corpus_chrf(finetuned_predictions, [val_references])



print(f'chrF AVANT fine-tuning : {baseline_chrf.score:.2f}')

print(f'chrF APRES fine-tuning : {finetuned_chrf.score:.2f}')

print()

print('Quelques exemples avant/apres :')

for i in range(5):

    print(f'  EN  : {val_sources[i]}')

    print(f'  REF : {val_references[i]}')

    print(f'  AVANT: {baseline_predictions[i]}')

    print(f'  APRES: {finetuned_predictions[i]}')

    print()

## 8. Sauvegarde de l'adaptateur LoRA

On ne sauvegarde QUE l'adaptateur LoRA (quelques dizaines de Mo), pas le modèle complet (2,4 Go) : c'est tout
l'intérêt du fine-tuning par adaptateur — petit fichier, facile à télécharger et à distribuer, à recombiner avec le
modèle de base au moment de l'utilisation.

In [ ]:
model.save_pretrained('./moore-nllb-lora-adapter')

tokenizer.save_pretrained('./moore-nllb-lora-adapter')



!zip -r moore-nllb-lora-adapter.zip moore-nllb-lora-adapter



from google.colab import files

files.download('moore-nllb-lora-adapter.zip')

## 9. Prochaine étape : intégrer l'adaptateur dans l'app

Une fois le zip téléchargé et décompressé dans `backend/models/moore-nllb-lora-adapter/` :

```python
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM

base_model = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M')
model = PeftModel.from_pretrained(base_model, 'backend/models/moore-nllb-lora-adapter')
```

Ce modèle `model` remplace alors le modèle de base dans `translation_service.py`, uniquement pour les requêtes
impliquant l'anglais et le mooré (le routage "pivot" français->anglais->mooré décrit en introduction reste à
implémenter côté API : deux appels de traduction à la chaîne au lieu d'un seul lorsque source ou cible = mooré et
que l'autre langue = français).